# Tournament simulation

## 1. One match
I simulate a match many times from the Poisson expected goals. In knockout games, a level score goes to extra time (one third of the expected goals), then to a penalty shootout.
Shootouts are a coin flip: since 2000, the higher-rated team wins 53.8% of 355 shootouts, which is within 1.4 standard errors of 50%.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.database import query
from src.models.poisson import PoissonModel
from src.simulation import simulate_knockout, simulate_scores

history = query("""
    SELECT date, home_team, away_team, home_score, away_score, neutral
    FROM clean_results
    ORDER BY date, match_id
""")
model = PoissonModel().fit(history, "2026-06-11")

In [2]:
final = pd.DataFrame({"home_team": ["Spain"], "away_team": ["Argentina"], "neutral": [True]})
home_goals, away_goals = model.expected_goals(final)
exact = model.predict_proba(final)[0]

rng = np.random.default_rng(42)
home, away = simulate_scores(home_goals[0], away_goals[0], 100_000, rng)
pd.DataFrame({
    "exact": exact,
    "simulated": [(home > away).mean(), (home == away).mean(), (home < away).mean()],
}, index=["Spain win", "draw", "Argentina win"]).round(3)

,exact,simulated
Spain win,0.281,0.282
draw,0.318,0.320
Argentina win,0.401,0.398


In [3]:
rng = np.random.default_rng(42)
knockout = simulate_knockout(home_goals[0], away_goals[0], 100_000, rng)
{name: round(values.mean(), 3) for name, values in knockout.items()}

{'home_wins': np.float64(0.433),
 'extra_time': np.float64(0.32),
 'shootout': np.float64(0.19)}

With 100,000 simulations, the simulated outcome shares match the exact Poisson probabilities within 0.3 points, the expected Monte Carlo error.
Before the 2026 World Cup, the Poisson model saw Argentina as the favourite in a final against Spain (40% vs 28% after 90 minutes), because of Argentina's very strong defence. In a knockout game, Spain would still go through 43% of the time: extra time in 32% of games, a shootout in 19%.
The actual final was 0-0 after 90 minutes and Spain won in extra time, one of the most likely paths under the model.

In [4]:
for as_of in ["2026-06-11", "2026-07-19"]:
    model_at_date = PoissonModel().fit(history, as_of)
    home_goals, away_goals = model_at_date.expected_goals(final)
    knockout = simulate_knockout(home_goals[0], away_goals[0], 100_000, np.random.default_rng(42))
    print(as_of, home_goals.round(2), away_goals.round(2),
          model_at_date.predict_proba(final).round(3), knockout["home_wins"].mean().round(3))

2026-06-11 [0.82] [1.05] [[0.281 0.318 0.401]] 0.433
2026-07-19 [0.95] [0.91] [[0.352 0.322 0.326]] 0.514


A tournament forecast uses the model fitted before the tournament for every match, even the final, five weeks later. A match-by-match forecast refits the model the day before each game. Refitted on 19 July with all World Cup results, the model makes Spain a slight favourite (35% vs 33%, 51% to win the final), close to the 0-0 and extra-time win that happened.

## 2. Group stage
I simulate the 12 groups 10,000 times with the model fitted on 11 June 2026. Ties are broken by goal difference, goals scored, then at random. The 8 best third-placed teams also qualify.
Host nations get the Poisson home advantage in their group matches (about 26% more goals).

In [5]:
from src.tournament import goal_rates, load_config, simulate_group_stage

config = load_config("world_cup_2026")
teams = [team for members in config["groups"].values() for team in members]
team_index = {team: i for i, team in enumerate(teams)}
rates, home_factor = goal_rates(model, teams)
n_sims = 10_000
positions, best_thirds = simulate_group_stage(config, team_index, rates, home_factor, n_sims, np.random.default_rng(42))

In [6]:
def share(team_ids):
    return np.bincount(team_ids.ravel(), minlength=len(teams)) / n_sims

first = sum(share(positions[f"1{letter}"]) for letter in config["groups"])
top_two = first + sum(share(positions[f"2{letter}"]) for letter in config["groups"])
qualified_thirds = sum(
    share(positions[f"3{letter}"][(best_thirds == letter).any(axis=1)]) for letter in config["groups"]
)
group_stage = pd.DataFrame({
    "group": [letter for letter, members in config["groups"].items() for _ in members],
    "first": first,
    "top_two": top_two,
    "qualified": top_two + qualified_thirds,
}, index=teams)
group_stage.sort_values("qualified", ascending=False).round(3)

,group,first,top_two,qualified
Spain,H,0.745,0.960,0.989
Brazil,C,0.595,0.892,0.984
England,L,0.666,0.936,0.984
Argentina,J,0.690,0.910,0.980
Germany,E,0.529,0.829,0.979
Switzerland,B,0.549,0.891,0.968
Belgium,G,0.610,0.860,0.955
Morocco,C,0.319,0.765,0.941
Canada,B,0.390,0.829,0.937
Ecuador,E,0.312,0.712,0.937
